# survey-gauge with External APIs (OpenAI + Hugging Face)

This notebook shows how to evaluate an LLM over a questionnaire using async clients wrapped with Instructor.

In [ ]:
%pip install -q -U instructor openai huggingface_hub pydantic pyyaml tqdm
# Optional if survey-gauge is not already installed:
# %pip install -q -U survey-gauge

In [ ]:
import os
import logging
from pathlib import Path

from survey_gauge import Questionnaire, Scenario, Eval

import instructor
from openai import AsyncOpenAI

In [ ]:
# Self-contained sample data so the notebook runs on Colab.
Path('classification_demo.yml').write_text('''preamble:
  Read the scenario and choose one label.
default_choices:
  - choice: low_risk
    score: 0
  - choice: high_risk
    score: 1
failure_indicator: N/A
questions:
  - question: Is this scenario high risk?
''', encoding='utf-8')

Path('scenarios_demo.yml').write_text('''- description: The subject plans a calm walk in the park.
- description: The subject is trapped in a burning building with no visible exit.
''', encoding='utf-8')

questionnaire = Questionnaire.from_yml('classification_demo.yml')
scenarios = Scenario.from_yml('scenarios_demo.yml')

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('Evaluation')

## OpenAI Example

Set your key first: `OPENAI_API_KEY`.

In [ ]:
MODEL_OPENAI = 'gpt-4o-mini'

openai_client = instructor.from_openai(
    AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY']),
    mode=instructor.Mode.JSON,
)

async def openai_call(prompt, output_model):
    return await openai_client.chat.completions.create(
        model=MODEL_OPENAI,
        messages=[{'role': 'user', 'content': prompt}],
        response_model=output_model,
        temperature=0.0,
        top_p=0.01,
        seed=42,
    )

evaluator_openai = Eval(openai_call, questionnaire=questionnaire, logger=logger)
scores_openai = await evaluator_openai.evaluate_scenario(scenarios[0])
print('OpenAI scores:', scores_openai)

## Hugging Face Router Example

Set your token first: `HF_TOKEN`.

In [ ]:
MODEL_HF = 'meta-llama/Llama-3.1-8B-Instruct'

hf_client = instructor.from_openai(
    AsyncOpenAI(
        base_url='https://router.huggingface.co/openai/v1',
        api_key=os.environ['HF_TOKEN'],
    ),
    mode=instructor.Mode.JSON,
)

async def hf_call(prompt, output_model):
    return await hf_client.chat.completions.create(
        model=MODEL_HF,
        messages=[{'role': 'user', 'content': prompt}],
        response_model=output_model,
        temperature=0.0,
        top_p=0.01,
        seed=42,
    )

evaluator_hf = Eval(hf_call, questionnaire=questionnaire, logger=logger)
scores_hf = await evaluator_hf.evaluate_scenario(scenarios[1])
print('HF scores:', scores_hf)